# AI와 함께 타이타닉 데이터 분석 한 사이클 완주하기

데이터 → 전처리 → 시각화 → EDA → Feature → 분류 → 평가 → 모델 선택 → 저장 → Streamlit

> 코드는 AI의 도움을 받아 최소한으로 작성하지만, 분석을 단계별로 진행하고 실제 결과를 보고 다음 행동을 결정하는 사람은 학생입니다.

**사용 원칙**: 실행 계획 → 코드 실행 → 실행 결과 요약 → 실행 결과 분석 순서를 지킵니다. 관찰/해석/가설/한계를 구분하고, AI가 실제 실행 결과를 만들어내게 하지 않습니다.


## STEP 00. 전체 분석 지도

### 실행 계획
전체 흐름과 AI/학생 역할을 확인한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


## STEP 01. 실행 환경 확인

### 실행 계획
현재 Notebook을 실행하는 Python 경로·버전·작업 경로와 주요 패키지 버전을 확인한다.

VS Code에서 선택한 **Python Interpreter**는 터미널에서 사용할 환경이고, Notebook 오른쪽 위의 **Kernel**은 셀을 실행할 환경이다. 둘이 다를 수 있으므로 Kernel에서도 사용할 Python 환경을 선택하고, 아래 `sys.executable` 출력으로 확인한다.

두 셀을 위에서부터 직접 실행한다. 패키지 import 오류가 나면 오류에 나온 패키지와 선택한 Kernel을 확인한다. 필요한 설치는 README 안내에 따라 같은 환경의 터미널에서 직접 수행한 뒤 Kernel을 다시 시작한다. 이 Notebook은 패키지를 자동 설치하거나 오류를 숨기지 않는다.


In [1]:
# 현재 셀을 실행하는 Python과 작업 경로
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Working directory:", Path.cwd())


Python executable: C:\dev\ai-data-analysis\.venv\Scripts\python.exe
Python version: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Working directory: C:\dev\llm-data-analysis-course\notebooks


In [2]:
# 패키지 버전: 오류가 나면 선택한 Kernel의 환경을 확인하세요.
import pandas as pd
import numpy as np
import sklearn
from IPython.display import display

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)


pandas: 3.0.5
numpy: 2.5.1
scikit-learn: 1.9.0


### 실행 결과 요약 — 학생 작성
- 현재 Python:
- 현재 Kernel:
- 주요 패키지:

### 실행 결과 분석 — 학생 작성
- 관찰:
- 해석:
- 가설 / 추가 확인:
- 한계:


## STEP 02. Titanic 데이터 준비와 로딩

### 실행 계획
저장소 루트의 터미널에서 Notebook Kernel과 같은 Python 환경으로 다음 명령을 먼저 실행한다.

```powershell
python scripts/prepare_titanic_data.py
```

준비 스크립트는 공개 원본을 다운로드하고 강의 데이터 기준으로 검증한다. 아래 helper는 **data 폴더만 기준으로** 현재 위치부터 상위 폴더를 탐색하므로 저장소 루트와 `notebooks/`에서 모두 사용할 수 있다.

경로를 확인한 뒤 CSV를 읽고 앞부분·크기·컬럼을 직접 확인한다. `df`는 이후 전체 실습의 원본 기준 DataFrame이며 이 단계에서는 전처리하지 않는다.


In [3]:
from pathlib import Path

def get_project_root(start_path: Path | None = None) -> Path:
    """현재 위치에서 상위 폴더를 탐색해 data 폴더가 있는 프로젝트 루트를 찾는다."""
    current = (start_path or Path.cwd()).resolve()

    for path in (current, *current.parents):
        if (path / "data").is_dir():
            return path

    raise FileNotFoundError(
        "data 폴더가 있는 프로젝트 루트를 찾을 수 없습니다."
    )

project_root = get_project_root()
data_path = project_root / "data" / "titanic" / "train.csv"

if not data_path.is_file():
    raise FileNotFoundError(
        f"Titanic 데이터가 없습니다: {data_path}\n"
        f"저장소 루트({project_root})에서 Kernel과 같은 Python 환경으로 "
        "python scripts/prepare_titanic_data.py 를 실행한 뒤 다시 실행하세요."
    )

print("Data path:", data_path)


Data path: C:\dev\llm-data-analysis-course\data\titanic\train.csv


In [4]:
df = pd.read_csv(data_path)

display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


### 실행 결과 요약 — 학생 작성
- 읽은 파일 경로:
- 행/열 수:
- 컬럼과 앞부분에서 확인한 내용:

### 실행 결과 분석 — 학생 작성
- 관찰:
- 해석:
- 가설 / 추가 확인:
- 한계:


## STEP 03. 데이터 구조와 품질의 첫인상

### 실행 계획
원본 `df`를 그대로 사용해 네 개의 작은 셀을 순서대로 실행한다. 각 출력에서 직접 확인한 사실을 적고, 전처리 방법은 아직 확정하지 않는다.

A: 행/열·컬럼·앞부분·dtype → B: 결측치와 중복 → C: 주요 범주형 값 → D: 기초 통계.

숫자로 저장된 식별자·범주 코드의 평균을 연속형 측정값처럼 해석하지 않도록 주의한다. 통계는 결측값을 제외해 계산되므로 `count`도 함께 살펴본다.


In [5]:
# A. 데이터 구조와 dtype
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
df.info()


Shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.7 KB


In [6]:
# B. 결측치 비율은 백분율(%)입니다.
missing_count = df.isna().sum()
missing_rate = df.isna().mean().mul(100)
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_rate": missing_rate.round(2),
})
display(missing_summary)
print("Duplicate rows:", df.duplicated().sum())


,missing_count,missing_rate
PassengerId,0,0.00
Survived,0,0.00
Pclass,0,0.00
Name,0,0.00
Sex,0,0.00
Age,177,19.87
SibSp,0,0.00
Parch,0,0.00
Ticket,0,0.00
Fare,0,0.00


Duplicate rows: 0


In [7]:
# C. 숫자로 저장된 Target/등급도 범주별로 관찰합니다.
for column in ["Survived", "Pclass", "Sex", "Embarked"]:
    print(f"Category: {column} (결측값 포함)")
    display(df[column].value_counts(dropna=False).rename("count").to_frame())


Category: Survived (결측값 포함)


,count
Survived,
0,549
1,342


Category: Pclass (결측값 포함)


,count
Pclass,
3,491
1,216
2,184


Category: Sex (결측값 포함)


,count
Sex,
male,577
female,314


Category: Embarked (결측값 포함)


,count
Embarked,
S,644
C,168
Q,77
NaN,2


In [8]:
# D. 숫자형과 문자형을 나누어 요약합니다. 원본 df는 바뀌지 않습니다.
print("Numeric summary:")
display(df.describe())
print("Text summary:")
display(df.select_dtypes(include=["object", "string"]).describe())


Numeric summary:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Text summary:


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


### 출력에서 생각해 볼 질문
- Age 결측치는 어떻게 할 것인가?
- Cabin은 결측치가 많아 보이는가?
- Embarked의 결측치는 많지 않은가?
- 숫자처럼 보여도 실제 의미는 범주형인 컬럼이 있는가?
- PassengerId는 예측 Feature로 의미가 있는가?
- Name, Ticket, Cabin은 바로 버려야 하는가?

아직 결측치 대체·행 삭제·컬럼 제거를 실행하지 않는다. `df_work`는 STEP 05, `df_encoded`는 STEP 07, `model_source`는 STEP 11에서 생성한다.


### 실행 결과 요약 — 학생 작성
- 데이터 구조:
- 결측치와 중복:
- 범주와 기초 통계에서 확인한 내용:

### 실행 결과 분석 — 학생 작성
- 관찰:
- 해석:
- 가설 / 추가 확인:
- 한계:


### 첫 번째 개인 판단 — 학생이 선택
실제 출력을 보고 **추가로 확인하고 싶은 데이터 품질 항목 하나**를 선택한다. 예: 이상치, 범주별 개수, 중복 PassengerId, Fare 분포, Age 범위, 특정 컬럼 unique 개수. AI가 대신 확정하지 않는다.

필요하면 실제 출력과 함께 다음 Prompt로 후보만 요청한다.

> 현재 STEP 03 실행 결과를 기준으로 추가로 확인할 가치가 있는 데이터 품질 분석 후보 3개를 제안해 주세요. 각 후보에 대해 1. 무엇을 확인하는지 2. 왜 필요한지 3. 결과에 따라 다음 판단이 어떻게 달라지는지를 설명해 주세요. 아직 코드는 작성하지 마세요.

- 내가 선택한 항목:
- 선택 이유:
- 진행하지 않은 후보와 이유:
- 추가 확인 후 기록할 실제 결과:
- 다음 판단:


## STEP 04. 분석 문제와 Target 정의

### 실행 계획
`Survived` 이진 분류 문제와 Feature/누수 후보를 정의한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [9]:
# STEP 04
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 05. 결측치 처리

### 실행 계획
`df_work = df.copy()`를 한 번 만들고 처리 후보를 비교한 뒤 학생이 선택한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [10]:
# STEP 05
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 06. 불필요한 컬럼 검토

### 실행 계획
기존 `df_work`를 이어서 유지/제외 후보/파생 후보/보류를 판단한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [11]:
# STEP 06
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 07. 범주형 데이터 변환 원리

### 실행 계획
`df_encoded = df_work.copy()`로 인코딩 원리만 연습한다. 최종 모델 입력으로 사용하지 않는다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [12]:
# STEP 07
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 08. 시각화와 기본 패턴 확인

### 실행 계획
`df_work`로 질문 → 그래프 → 관찰 → 해석을 수행한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [13]:
# STEP 08
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 09. 기초 통계와 EDA

### 실행 계획
같은 `df_work`로 비율·표본 수·평균/중앙값·그룹 비교를 수행한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [14]:
# STEP 09
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 10. Feature 설계

### 실행 계획
결정적 파생 Feature와 데이터에서 기준을 학습하는 Feature를 구분한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [15]:
# STEP 10
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 11. 학습/테스트 데이터 준비

### 실행 계획
원본 `df`에서 `model_source`를 다시 만들고 전처리보다 먼저 split한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [16]:
# STEP 11
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 12. Baseline 분류 모델 학습

### 실행 계획
`ColumnTransformer + Pipeline + LogisticRegression`을 `X_train`에만 fit한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [17]:
# STEP 12
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 13. 모델 성능 평가와 오류 분석

### 실행 계획
Accuracy·Confusion Matrix·Precision·Recall·F1·오분류 사례를 확인한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [18]:
# STEP 13
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 14. 추가 모델/알고리즘 선택

### 실행 계획
학생이 후보 하나를 선택하고 같은 전처리 계약의 `additional_pipeline`을 만든다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [19]:
# STEP 14
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 15. 모델 비교와 최종 모델 선정

### 실행 계획
train-only CV를 보강 근거로 사용하고 `final_pipeline`을 확정한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [20]:
# STEP 15
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 16. 최종 Pipeline 저장과 새 승객 예측

### 실행 계획
Pipeline/계약을 저장하고 재로딩 예측 일치를 검증한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [21]:
# STEP 16
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 17. Streamlit 예측 앱 구현

### 실행 계획
저장 Pipeline을 그대로 사용해 입력 → 예측 → 확률 표시를 연결한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [22]:
# STEP 17
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

# 최종 회고

- 가장 중요한 관찰:
- 내가 직접 내린 판단:
- AI 제안을 수정하거나 거절한 사례:
- 최종 모델과 선택 이유:
- 현재 모델/앱의 한계:
- 다음 확장: 데이터 수집 자동화 → 분석 자동화 → 보고서 자동화 → 사람 승인 기반 Data Analysis Agent
